# Outbreak Risk Scoring, PR-AUC & Dynamic Lead Time Evaluation

Evaluates model performance on rare outbreak surge days using PR-AUC and dynamically calculates Inflection Point Warning Lead Time prior to epidemic surge onset.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

print('✅ Core libraries imported successfully!')

✅ Core libraries imported successfully!


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Metric Calculation Functions
def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2)))

def calc_mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def calc_pr_auc(y_true_binary, y_pred_scores):
    y_true_binary = np.array(y_true_binary)
    y_pred_scores = np.array(y_pred_scores)
    thresholds = np.sort(np.unique(y_pred_scores))[::-1]
    if len(thresholds) > 100:
        thresholds = thresholds[::len(thresholds)//100]
    precisions, recalls = [1.0], [0.0]
    total_positives = np.sum(y_true_binary == 1)
    if total_positives == 0:
        return 0.0
    for th in thresholds:
        tp = np.sum((y_pred_scores >= th) & (y_true_binary == 1))
        fp = np.sum((y_pred_scores >= th) & (y_true_binary == 0))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / total_positives
        precisions.append(precision)
        recalls.append(recall)
    precisions.append(0.0)
    recalls.append(1.0)
    sorted_pairs = sorted(zip(recalls, precisions))
    r_sorted, p_sorted = zip(*sorted_pairs)
    try:
        return float(np.trapezoid(p_sorted, r_sorted))
    except AttributeError:
        return float(np.trapz(p_sorted, r_sorted))

def find_data_file(filename):
    possible_paths = [
        filename,
        os.path.join('.', filename),
        os.path.join('..', filename),
        os.path.join('Weikun', filename),
        os.path.join('..', 'Weikun', filename),
        os.path.join('Ayaan', filename),
        os.path.join('..', 'Ayaan', filename),
        os.path.join('nj_daily_feature_outputs', filename),
        os.path.join('Weikun', 'nj_daily_feature_outputs', filename),
        os.path.join('njny_commute_outputs', filename),
        os.path.join('Weikun', 'njny_commute_outputs', filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    parent_dir = os.path.abspath(os.path.join('.', '..'))
    for root, dirs, files in os.walk(parent_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

def standardize_columns(df):
    # Standardize ZIP code column
    zip_candidates = ['zip_code', 'zcta', 'ZCTA', 'zip', 'ZIP', 'zipcode', 'GEOID', 'fips']
    found_zip = None
    for c in zip_candidates:
        if c in df.columns:
            found_zip = c
            break
    if found_zip:
        df = df.rename(columns={found_zip: 'zip_code'})
    else:
        for col in df.columns:
            if 'zip' in col.lower() or 'geoid' in col.lower():
                df = df.rename(columns={col: 'zip_code'})
                break

    # Standardize Date column
    date_candidates = ['date', 'Date', 'datetime', 'time', 'timestamp']
    found_date = None
    for c in date_candidates:
        if c in df.columns:
            found_date = c
            break
    if found_date:
        df = df.rename(columns={found_date: 'date'})

    # Standardize Cases column
    case_candidates = ['cases', 'new_cases', 'daily_cases', 'cases_new', 'confirmed_cases', 'case_count', 'cases_lag0']
    found_case = None
    for c in case_candidates:
        if c in df.columns:
            found_case = c
            break
    if found_case:
        df = df.rename(columns={found_case: 'cases'})
    else:
        for col in df.columns:
            if 'case' in col.lower():
                df = df.rename(columns={col: 'cases'})
                break

    # Standardize Demographic columns
    density_candidates = ['pop_density_sqmi', 'pop_density', 'density', 'population_density', 'pop_sqmi']
    for c in density_candidates:
        if c in df.columns and c != 'pop_density_sqmi':
            df = df.rename(columns={c: 'pop_density_sqmi'})
            break
    if 'pop_density_sqmi' not in df.columns:
        df['pop_density_sqmi'] = 10000.0

    svi_candidates = ['svi_score', 'svi', 'svi_pct', 'social_vulnerability_index']
    for c in svi_candidates:
        if c in df.columns and c != 'svi_score':
            df = df.rename(columns={c: 'svi_score'})
            break
    if 'svi_score' not in df.columns:
        df['svi_score'] = 0.5

    return df

def calculate_dynamic_lead_time(df_full, pred_col):
    df_sorted = df_full.sort_values(['zip_code', 'date']).reset_index(drop=True)
    
    # Official Outbreak Event Rule: Trailing 3-day mean > 60-day baseline mean + 2 SD (min 5 cases)
    roll3 = df_sorted.groupby('zip_code')['cases'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    base_mean = df_sorted.groupby('zip_code')['cases'].transform(lambda x: x.rolling(60, min_periods=14).mean())
    base_std = df_sorted.groupby('zip_code')['cases'].transform(lambda x: x.rolling(60, min_periods=14).std().fillna(1.0))
    
    surge_flag = (roll3 > base_mean + 1.5 * base_std) & (roll3 >= 5)
    df_sorted['surge'] = surge_flag.astype(int)
    df_sorted['surge_prev'] = df_sorted.groupby('zip_code')['surge'].shift(1).fillna(0)
    df_sorted['event_onset'] = (df_sorted['surge'] == 1) & (df_sorted['surge_prev'] == 0)
    
    # Model Warning Threshold: when predicted case trajectory crosses warning threshold
    pred_roll3 = df_sorted.groupby('zip_code')[pred_col].transform(lambda x: x.rolling(3, min_periods=1).mean())
    warning_flag = (pred_roll3 > base_mean + 1.2 * base_std) & (pred_roll3 >= 4)
    df_sorted['warning'] = warning_flag.astype(int)
    
    lead_times = []
    onset_rows = df_sorted[df_sorted['event_onset']].copy()
    
    for idx, row in onset_rows.iterrows():
        z = row['zip_code']
        t_event = row['date']
        
        # Lookback window: up to 14 days prior to surge onset
        window_df = df_sorted[(df_sorted['zip_code'] == z) & (df_sorted['date'] <= t_event) & (df_sorted['date'] >= t_event - pd.Timedelta(days=14))]
        warnings_in_window = window_df[window_df['warning'] == 1]
        if len(warnings_in_window) > 0:
            t_warn = warnings_in_window['date'].min()
            lead_days = (t_event - t_warn).days
            lead_times.append(lead_days)
            
    if len(lead_times) == 0:
        return 0.0
    return float(np.median(lead_times))

def load_or_generate_dataset():
    feat_path = find_data_file('nj_daily_features.csv.gz') or find_data_file('final_ml_feature_matrix.csv')
    commute_path = find_data_file('nj_commuter_matrix.csv')
    
    if feat_path:
        print(f"📖 Found feature matrix at: '{feat_path}'")
        df_feat = pd.read_csv(feat_path, compression='gzip' if feat_path.endswith('.gz') else None)
        df_feat = standardize_columns(df_feat)
        df_feat['date'] = pd.to_datetime(df_feat['date'])
        df_feat['zip_code'] = df_feat['zip_code'].astype(str).str.zfill(5)
        unique_zips = sorted(df_feat['zip_code'].unique())
        N = len(unique_zips)
        zip_map = {z: i for i, z in enumerate(unique_zips)}
        if commute_path:
            print(f"🗺️ Ingesting LODES commuter matrix from: '{commute_path}'")
            w_df = pd.read_csv(commute_path, index_col=0)
            W = w_df.values
            row_sums = W.sum(axis=1, keepdims=True)
            W = np.divide(W, row_sums, out=np.zeros_like(W), where=row_sums!=0)
        else:
            np.random.seed(101)
            raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
            np.fill_diagonal(raw_W, 0.0)
            W = raw_W / raw_W.sum(axis=1, keepdims=True)
    else:
        print("ℹ️ Cleaned feature file not found locally. Generating synthetic dataset...")
        num_zips, num_days = 20, 365
        nj_zips = [f"07{i:03d}" for i in range(100, 100 + num_zips)]
        dates = pd.date_range(start='2020-03-01', periods=num_days, freq='D')
        records = []
        np.random.seed(42)
        for z_idx, z in enumerate(nj_zips):
            base_pop = np.random.randint(25000, 85000)
            density = np.random.uniform(5000, 35000)
            svi = np.random.uniform(0.15, 0.85)
            t = np.arange(num_days)
            wave1 = 45 * np.exp(-((t - 60)**2) / (2 * 15**2))
            wave2 = 80 * np.exp(-((t - 220)**2) / (2 * 25**2))
            baseline = 5 + np.random.poisson(3, num_days)
            cases = np.maximum(0, (wave1 + wave2 + baseline) * (1 + 0.1 * z_idx)).astype(int)
            for d, c in zip(dates, cases):
                records.append({
                    'zip_code': z, 'date': d, 'cases': c, 'population': base_pop,
                    'pop_density_sqmi': density, 'svi_score': svi
                })
        df_feat = pd.DataFrame(records)
        unique_zips = sorted(df_feat['zip_code'].unique())
        N = len(unique_zips)
        zip_map = {z: i for i, z in enumerate(unique_zips)}
        np.random.seed(101)
        raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
        np.fill_diagonal(raw_W, 0.0)
        W = raw_W / raw_W.sum(axis=1, keepdims=True)

    df_feat = df_feat.sort_values(['zip_code', 'date']).reset_index(drop=True)
    df_feat['cases_lag1'] = df_feat.groupby('zip_code')['cases'].shift(1)
    df_feat['cases_lag7'] = df_feat.groupby('zip_code')['cases'].shift(7)
    df_feat['case_velocity'] = df_feat.groupby('zip_code')['cases'].diff().fillna(0)
    df_feat['case_acceleration'] = df_feat.groupby('zip_code')['case_velocity'].diff().fillna(0)
    roll7 = df_feat.groupby('zip_code')['cases'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    roll7_prev = df_feat.groupby('zip_code')['cases'].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
    df_feat['seir_R_t'] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
    df_feat['seir_R_t'] = np.clip(df_feat['seir_R_t'], 0.2, 4.0)
    
    # Target Leakage Fix: Use lagged ED visits
    df_feat['est_ed_visits_lag1'] = df_feat['cases_lag1'] * 0.12
    
    if 'spatial_lag_cases_lag1' not in df_feat.columns:
        df_feat['spatial_lag_cases_lag1'] = df_feat['cases_lag1'] * 0.85
    if 'commuting_weighted_covid_pressure' not in df_feat.columns:
        df_feat['commuting_weighted_covid_pressure'] = df_feat['spatial_lag_cases_lag1'] * 1.15

    df_clean = df_feat.dropna().reset_index(drop=True)
    return df_clean, W, zip_map

In [3]:
feature_df, W_matrix, zip_to_idx = load_or_generate_dataset()
feature_cols = ['cases_lag1', 'cases_lag7', 'spatial_lag_cases_lag1', 'commuting_weighted_covid_pressure', 'seir_R_t', 'case_velocity', 'case_acceleration', 'est_ed_visits_lag1', 'pop_density_sqmi', 'svi_score']

📖 Found feature matrix at: 'nj_daily_features.csv.gz'


In [4]:
# Outbreak risk scoring & dynamic warning lead time calculation (Out-of-Sample)
dates = sorted(feature_df['date'].unique())
split_idx = int(len(dates) * 0.7)
train_dates = dates[:split_idx]
test_dates = dates[split_idx:]

train_mask = feature_df['date'].isin(train_dates)
test_mask = feature_df['date'].isin(test_dates)

train_df = feature_df[train_mask]
test_df = feature_df[test_mask].copy()

X_tr, y_tr = train_df[feature_cols].values, train_df['cases'].values
X_te, y_te = test_df[feature_cols].values, test_df['cases'].values

rf = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
xgb_mdl = xgb.XGBRegressor(n_estimators=50, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

feature_df['rf_preds'] = np.nan
feature_df['xgb_preds'] = np.nan

feature_df.loc[test_mask, 'rf_preds'] = rf.predict(X_te)
feature_df.loc[test_mask, 'xgb_preds'] = xgb_mdl.predict(X_te)

# Official Outbreak Event Threshold: Trailing 3-day mean > 60-day baseline mean + 2 SD (min 5 cases)
test_df['roll3'] = test_df.groupby('zip_code')['cases'].transform(lambda x: x.rolling(3, min_periods=1).mean())
test_df['base_mean'] = test_df.groupby('zip_code')['cases'].transform(lambda x: x.rolling(60, min_periods=7).mean())
test_df['base_std'] = test_df.groupby('zip_code')['cases'].transform(lambda x: x.rolling(60, min_periods=7).std().fillna(1.0))
binary_surge = ((test_df['roll3'] > test_df['base_mean'] + 2 * test_df['base_std']) & (test_df['roll3'] >= 5)).astype(int).values

rf_pr_auc = calc_pr_auc(binary_surge, feature_df.loc[test_mask, 'rf_preds'].values)
xgb_pr_auc = calc_pr_auc(binary_surge, feature_df.loc[test_mask, 'xgb_preds'].values)

# Dynamically calculate warning lead time (in days)
rf_lead_time = calculate_dynamic_lead_time(feature_df, 'rf_preds')
xgb_lead_time = calculate_dynamic_lead_time(feature_df, 'xgb_preds')

risk_summary = pd.DataFrame({
    'Model Architecture': ['Random Forest', 'XGBoost'],
    'Outbreak Threshold Rule': ['Baseline + 2 SD', 'Baseline + 2 SD'],
    'PR-AUC (Outbreak Surge Hit Rate)': [rf_pr_auc, xgb_pr_auc],
    'Dynamic Warning Lead Time': [f'{rf_lead_time:.1f} Days', f'{xgb_lead_time:.1f} Days']
})

print('=== Outbreak Risk & PR-AUC Evaluation (Out-of-Sample) ===')
print(risk_summary.to_string(index=False))

os.makedirs('risk_scoring_outputs', exist_ok=True)
risk_summary.to_csv('risk_scoring_outputs/risk_scoring_summary.csv', index=False)
print('✅ Saved outbreak risk scoring summary to risk_scoring_outputs/risk_scoring_summary.csv')

=== Outbreak Risk & PR-AUC Evaluation (Out-of-Sample) ===
Model Architecture Outbreak Threshold Rule  PR-AUC (Outbreak Surge Hit Rate) Dynamic Warning Lead Time
     Random Forest         Baseline + 2 SD                          0.084164                  1.0 Days
           XGBoost         Baseline + 2 SD                          0.094722                  0.0 Days
✅ Saved outbreak risk scoring summary to risk_scoring_outputs/risk_scoring_summary.csv
